# Step 4 · The final exam and tricky sentences

**Pain–Comfort Lab · Looking inside a language model** · ⏱️ about 30 minutes · ⚡ GPU recommended

> ### 🤔 Wonder
> Our ruler worked on sentences it was built from. Does it work on sentences it has **never seen**? And what does it do with sentences that aren't simply pain *or* comfort, like *"He felt a burning desire to win"*?

**In this step you will:**

1. Open the sealed test set **once** and report an honest score, with a range of uncertainty.
2. Score neutral, mixed, metaphorical and negated sentences on a rescaled ruler.
3. Measure your own "mystery sentences".
4. Draw a 2-D "shadow" picture of the 1,536-dimensional snapshots (PCA).

## Setup and catch-up

In [ ]:
# 🔧 Setup: just run this cell. It works in Google Colab and in Jupyter on your own computer.
# It makes the lab's helper library (the `pclab` folder) and the sentence files available.
import subprocess, sys
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version

REPO_URL = 'https://github.com/23menon23/pain-comfort-lab'   # the lab's home on GitHub

if Path('../pclab').exists():                    # you are inside a downloaded copy of the repository
    LAB_ROOT = Path('..').resolve()
else:                                            # Google Colab: download the lab once per session
    LAB_ROOT = Path('pain-comfort-lab').resolve()
    if not LAB_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB_ROOT)], check=True)
sys.path.insert(0, str(LAB_ROOT))

if Version(version('transformers')) < Version('4.57'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57'], check=True)
    print('⚠️ Transformers was updated. Choose Runtime → Restart session, then run this cell again.')
else:
    import pclab
    from pclab import look, show_source
    print('✅ Ready. pclab', pclab.__version__, '| Transformers', version('transformers'))

In [ ]:
# 🔧 Load the model: just run this cell (the first download is about 3 GB and takes a few minutes).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Keep this model for class. 'Qwen/Qwen2.5-0.5B' uses less memory, but its results
# will differ, so record it as a different experiment.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

SEED = pclab.set_seed(42)          # fixes random choices so results can be repeated
DEVICE = pclab.device_report()     # 'cuda' (GPU) or 'cpu'
tokenizer, model = pclab.load_model(MODEL_NAME)
LAYERS = pclab.get_layers(model)
N_LAYERS = len(LAYERS)
HIDDEN_SIZE = model.config.hidden_size
pclab.use_lab_style()
print(f"Loaded {pclab.model_facts(model)['model']}: {N_LAYERS} layers, {HIDDEN_SIZE} numbers per token")

In [ ]:
# 🔧 Catch-up: rebuild Steps 2–3 (same sentences, same rule, so the same frozen layer)
state = pclab.rebuild_lab_state(model, tokenizer)
look(state, 'state')

### 📦 What came back?

`state` is a **dictionary**: a bundle of named results from Steps 2–3. Pulling items out by name keeps the rest of the notebook readable.

> **🧱 Why a dictionary instead of returning 13 separate values?** `a, b, c, d, ... = f()` with 13 names is easy to get in the wrong order, and a wrong order gives silent bugs. With a dictionary, every value has a label: `state['best_layer']` can't be confused with `state['vector_lengths']`.

Want to see how it was built? Run `show_source(pclab.rebuild_lab_state)`.

In [ ]:
# 🔧 Unpack the parts we need into ordinary variables
build_df, test_df = state['build_df'], state['test_df']
train_act, y_train = state['train_act'], state['y_train']
unit_vectors, midpoints = state['unit_vectors'], state['midpoints']
vector_lengths = state['vector_lengths']
BEST_LAYER = state['best_layer']
print(f'Frozen layer: {BEST_LAYER + 1} (Python index {BEST_LAYER})')

## Part 1 · The final exam: open the test set

Now we use the frozen ruler on the 16 test sentences, which played no part in building the ruler or choosing the layer.

**✏️ Predict first:** how many of the 16 will the ruler get right?

In [ ]:
# 🔧 Final test: run this cell once
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

test_act = pclab.collect(model, tokenizer, test_df['text'], 'Test snapshots')
y_test = test_df['label'].to_numpy()

test_scores = pclab.scores_at_all_layers(test_act, unit_vectors, midpoints)[:, BEST_LAYER]
test_predictions = (test_scores > 0).astype(int)
look(test_scores, 'test_scores', dims=('sentences',))
look(test_predictions, 'test_predictions', dims=('sentences',))

test_accuracy = accuracy_score(y_test, test_predictions)
test_auc = roc_auc_score(y_test, test_scores)
correct = int((test_predictions == y_test).sum())
lo, hi = pclab.wilson_interval(correct, len(y_test))

print(f'\nFrozen layer {BEST_LAYER + 1}: {correct}/{len(y_test)} correct')
print(f'Accuracy = {test_accuracy:.1%}   AUC = {test_auc:.3f}')
print(f'Plausible range for accuracy (rough 95% interval): {lo:.0%} to {hi:.0%}')

### 🔍 Code walk-through

| Code | Shape | What it does and why |
|---|---|---|
| `scores_at_all_layers(...)` | `(16, 28)` | every test sentence at every layer… |
| `[:, BEST_LAYER]` | `(16,)` | …then keep **one column**: the frozen layer. We never look at other layers' test scores. |
| `test_scores > 0` | `(16,)` True/False | which side of the midpoint |
| `.astype(int)` | `(16,)` 0/1 | turns True/False into 1/0 so it matches the labels |
| `(test_predictions == y_test).sum()` | one number | True counts as 1, so the sum is the number correct |
| `wilson_interval(correct, total)` | two numbers | a **range** of accuracies that are plausible given only 16 sentences |

> **🎛️ Why report a range?** With 16 sentences, one more or one fewer correct moves accuracy by about 6 points. The interval says: "on more sentences like these, the true accuracy could reasonably be anywhere from *lo* to *hi*." (Advanced note: it assumes sentences are independent. Ours are hand-written pairs, so treat it as a rough guide.)

In [ ]:
# 🔧 The full report, and a confusion matrix
test_report = test_df[['pair_id', 'text', 'category']].copy()
test_report['score'] = test_scores.round(3)
test_report['prediction'] = np.where(test_predictions == 1, 'Comfort', 'Pain')
test_report['correct'] = test_predictions == y_test
with pd.option_context('display.max_colwidth', 80):
    display(test_report)

cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])
look(cm, 'confusion matrix', dims=('true label', 'predicted label'))
display(pd.DataFrame(cm, index=['True Pain', 'True Comfort'], columns=['Predicted Pain', 'Predicted Comfort']))

**Reading the confusion matrix:** rows are the truth, columns are the ruler's guess. The diagonal (top-left, bottom-right) counts correct answers. Off-diagonal cells show *which kind* of mistake happened. Did the ruler call pain "comfort", or the other way round? `labels=[0, 1]` fixes the order of the rows and columns so the table reads the same every time.

## Part 2 · Tricky sentences: neutral, mixed and metaphorical

Our ruler always puts every sentence on one side or the other, even sentences that have nothing to do with pain or comfort. So a score alone doesn't tell us a sentence is *about* pain or comfort.

To make scores easier to read, we **rescale** the ruler so that:

- the **average pain** build sentence sits at **−1**,
- the **average comfort** build sentence sits at **+1**,
- **0** is the midpoint.

Scores can go beyond ±1. Zero is just the midpoint, not a proven "neutral" point.

### ✏️ Predict first

Before running the next cells, label each challenge sentence yourself as *pain*, *comfort*, *mixed*, *neutral/metaphor*, or *unsure*.

In [ ]:
challenge = pclab.load_challenge()
neutral_sentences = pclab.load_neutral()
with pd.option_context('display.max_colwidth', 80):
    display(challenge[['challenge_id', 'text']])

In [ ]:
# 🔧 Score neutral and challenge sentences on the rescaled ruler
REFERENCE_SCALE = float(vector_lengths[BEST_LAYER]) / 2

def ruler_scores(activations):
    """Scores at the frozen layer, rescaled so pain average = −1 and comfort average = +1."""
    return pclab.scores_at_all_layers(activations, unit_vectors, midpoints)[:, BEST_LAYER] / REFERENCE_SCALE

neutral_act = pclab.collect(model, tokenizer, neutral_sentences, 'Neutral snapshots')
challenge_act = pclab.collect(model, tokenizer, challenge['text'], 'Challenge snapshots')

challenge_report = challenge.copy()
challenge_report['ruler_score'] = ruler_scores(challenge_act).round(2)
with pd.option_context('display.max_colwidth', 70):
    display(challenge_report)   # now the 'type' and 'what_makes_it_tricky' columns are revealed

### 🔍 Why divide by *half* the arrow's length?

The arrow runs from the pain average to the comfort average, and the midpoint sits exactly halfway. So the comfort average is **half an arrow-length** above zero on the ruler, and the pain average is half an arrow-length below. Dividing by that half-length puts them at exactly +1 and −1. Don't take that on trust; check it:

In [ ]:
build_ruler = ruler_scores(train_act)
print('Average comfort build sentence:', round(build_ruler[y_train == 1].mean(), 4))
print('Average pain build sentence:   ', round(build_ruler[y_train == 0].mean(), 4))

### Where does everything land?

**Look especially at:** the literal burn vs. the "burning desire" (metaphor), and *happy-but-hurting* vs. *sad-but-comfortable* (mood vs. sensation). With only eight sentences these can't settle anything, but they give first hints about whether the ruler tracks **physical sensation**, **general mood** or **word choice**.

In [ ]:
# 🔧 One picture: test sentences, neutral sentences and challenge sentences on the same ruler
C, M = pclab.COLORS, pclab.MARKERS
groups = [
    ('Pain test', ruler_scores(test_act[y_test == 0]), C['pain'], M['pain']),
    ('Neutral', ruler_scores(neutral_act), C['neutral'], M['neutral']),
    ('Challenge', ruler_scores(challenge_act), C['accent'], 'D'),
    ('Comfort test', ruler_scores(test_act[y_test == 1]), C['comfort'], M['comfort']),
]
rng = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(9, 4.5))
for i, (name, values, color, marker) in enumerate(groups):
    ax.scatter(i + rng.uniform(-0.08, 0.08, len(values)), values, color=color, marker=marker, s=55, alpha=0.85)
for x, y, cid in zip(np.full(len(challenge), 2) + 0.12, ruler_scores(challenge_act), challenge['challenge_id']):
    ax.annotate(str(cid), (x, y), fontsize=8, va='center')
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.axhline(1, color=C['comfort'], linestyle=':', linewidth=1)
ax.axhline(-1, color=C['pain'], linestyle=':', linewidth=1)
ax.set_xticks(range(len(groups)), [g[0] for g in groups])
ax.set(ylabel='Ruler score (pain avg = −1, comfort avg = +1)', title=f'Where do sentences land at layer {BEST_LAYER + 1}?')
plt.show()

Numbers next to the purple diamonds are the challenge IDs from the table above.

### 🧪 Mystery-sentence reader

Type any sentence, **write down your guess first**, then click **Measure**. This uses the same frozen ruler; your sentence doesn't change it.

Ideas: a pain word used happily (*"That workout hurt so good"*), a negation (*"The shot didn't hurt at all"*), emotional pain with no body (*"Her words hurt him deeply"*).

If the widget doesn't appear, use the backup line at the bottom of the cell.

In [ ]:
# 🧪 Experiment: type your own sentences
import ipywidgets as widgets
from IPython.display import clear_output
pclab.enable_widgets()

mystery_records = []

def inspect_sentence(text):
    score = float(ruler_scores(pclab.extract_one(model, tokenizer, text)[None, :, :])[0])
    mystery_records.append({'text': text, 'ruler_score': score})
    print(text)
    print(f'Score: {score:+.3f}   (pain avg = −1, comfort avg = +1)')
    print('Side of the midpoint:', 'comfort' if score > 0 else 'pain')
    print('(A position on our ruler, not a probability or a feeling.)')

sentence_box = widgets.Textarea(value=challenge['text'][4], description='Sentence:',
                                layout=widgets.Layout(width='95%', height='70px'))
measure_button = widgets.Button(description='Measure', button_style='info')
result_area = widgets.Output()

def on_measure(_):
    with result_area:
        clear_output(wait=True)
        if sentence_box.value.strip():
            inspect_sentence(sentence_box.value.strip())

measure_button.on_click(on_measure)
display(sentence_box, measure_button, result_area)

# Backup: if the widget doesn't appear, remove the # below and run this cell again.
# inspect_sentence('The shot didn\'t hurt at all.')

### 🔍 Why `[None, :, :]`?

`extract_one` returns one sentence's snapshot, shape `(28, 1536)`. But `scores_at_all_layers` expects a **batch** of sentences, shape `(n, 28, 1536)`. Indexing with `None` adds a new dimension of size 1 at the front: `(1, 28, 1536)`, a batch of one. The `[0]` at the end takes that single score back out. You'll meet this "wrap it as a batch of one, then unwrap" pattern all over machine-learning code.

## Part 3 · A picture of 1,536 dimensions (PCA)

We can't draw 1,536 dimensions, but we can draw a **shadow**. **PCA** (principal component analysis) finds the two directions in which the snapshots vary the most and flattens everything onto them, like the shadow of a 3-D object on a wall.

- PCA **doesn't know** which sentences are pain or comfort. It looks for the biggest differences of *any* kind.
- A shadow hides information. Overlapping dots don't mean the model can't tell sentences apart, and separated dots don't prove anything deep.

**✏️ Predict first:** could our ruler work well even if pain and comfort overlap in this picture?

In [ ]:
# 🔧 PCA built from the build set only, then the test and neutral sentences placed onto it
from sklearn.decomposition import PCA

pca = PCA(n_components=2, svd_solver='full')
pca.fit(train_act[:, BEST_LAYER, :])                      # learn the two directions from the build set
pca_test = pca.transform(test_act[:, BEST_LAYER, :])       # place the test sentences
pca_neutral = pca.transform(neutral_act[:, BEST_LAYER, :])
look(pca_test, 'pca_test', dims=('sentences', 'components'))

fig, ax = plt.subplots(figsize=(7, 5))
for label, name, key in [(0, 'Pain test', 'pain'), (1, 'Comfort test', 'comfort')]:
    mask = y_test == label
    ax.scatter(pca_test[mask, 0], pca_test[mask, 1], color=C[key], marker=M[key], s=65, label=name)
ax.scatter(pca_neutral[:, 0], pca_neutral[:, 1], color=C['neutral'], marker=M['neutral'], s=70, label='Neutral')
ax.set(xlabel='Principal component 1', ylabel='Principal component 2', title=f'PCA shadow · layer {BEST_LAYER + 1}')
ax.legend()
plt.show()

print('Share of the build set\'s variation kept by PC1 and PC2:', np.round(pca.explained_variance_ratio_, 3))
print(f'Total kept: {pca.explained_variance_ratio_.sum():.1%}')

### 🔍 Code walk-through

| Code | What it does | Why |
|---|---|---|
| `train_act[:, BEST_LAYER, :]` | all build sentences at the frozen layer, `(32, 1536)` | PCA works on a 2-D table: rows = items, columns = features |
| `pca.fit(...)` | **learns** the two most-varying directions | fit on the **build set only**: if test sentences shaped the picture, that would be a quiet form of leakage |
| `pca.transform(...)` | **places** new sentences onto those directions | returns `(sentences, 2)`: two coordinates per sentence, ready to plot |
| `explained_variance_ratio_` | how much of the spread each axis keeps | if it's small, the picture hides most of what's going on |

This **fit / transform** split is the same pattern used across scikit-learn: *fit* learns from data, *transform* (or *predict*) applies what was learned to new data.

### 🌱 Does our ruler line up with the biggest variation?

If pain vs. comfort were the *main* thing that varies in the snapshots, our direction would point along PC1. Cosine similarity tells us (1 = same direction, 0 = unrelated; the sign doesn't matter here).

In [ ]:
for i, component in enumerate(pca.components_, start=1):
    cos = float(np.dot(component, unit_vectors[BEST_LAYER]))   # both have length 1, so the dot product is the cosine
    print(f'Cosine between our direction and PC{i}: {cos:+.3f}')

## Save your results

This writes your tables and settings to a `results/` folder, zips it, and (in Colab) downloads it. Save your notebook separately with **File → Save a copy in Drive**.

In [ ]:
# 🔧 Save
pclab.save_results(
    'step4_test_and_tricky',
    tables={'test_report': test_report, 'challenge_scores': challenge_report,
            'mystery_sentences': pd.DataFrame(mystery_records, columns=['text', 'ruler_score']),
            'validation_layers': state['validation_results']},
    settings={**pclab.model_facts(model, MODEL_NAME), 'seed': SEED,
              'selected_layer_1_based': BEST_LAYER + 1, 'readout_suffix': pclab.READOUT_SUFFIX,
              'readout_location': 'layer output, last token, before final norm',
              'layer_selection': 'validation accuracy, then validation AUC, then earliest layer',
              'test_accuracy': test_accuracy, 'test_auc': test_auc,
              'test_accuracy_interval_95': [lo, hi]},
)

---

## 🧭 Explain it

1. Report your frozen layer, test accuracy (correct/total), AUC and plausible range. Why is the range so wide?
2. Pick three challenge or mystery sentences, including one metaphor or negation and one mixed sensation. For each: your prediction, the score, and a possible explanation.
3. For one surprising result, give **two different** possible explanations (for example: word clues, mixed sensations, general mood, the template, the small build set).
4. Could the ruler work well even if pain and comfort overlap in the PCA picture? Explain.

## 🌱 Curiosity branches

- **Metaphor hunt:** where do 20 metaphorical phrases land (*"a painful meeting", "a soft voice", "a warm welcome"*)?
- **Negation:** does the model handle *not*, or does it react to the word "hurt"? Write 10 negated sentences and compare them with their un-negated versions.
- **Emotional pain:** does heartbreak land on the same side as a burned hand?

**Next:** [Step 5 · Fair comparisons](05_fair_comparisons.ipynb)